# Day 7 Lab 07: Silver Quality and Quarantine

        **Layer:** Silver  
        **Python reference:** `Lab_Files/labs/lab_07_silver_quality_quarantine.py`

        This notebook is a sectioned companion version of the existing Python lab. It does not replace `run_lab.py` or the scripts under `Lab_Files/labs`.

        ## Learning Checkpoints
        - Apply data quality checks to cleaned orders.
- Split valid rows from invalid quarantine rows.
- Inspect quality error arrays before writing Silver outputs.

**Dependency:** Run Lab/Notebook 04 first so Bronze orders are available.

## 0. Notebook Setup

Run this first. It moves the kernel into `Lab_Files` and makes the shared lab helpers importable.

In [ ]:
from pathlib import Path
import os
import sys

HERE = Path.cwd().resolve()
LAB_FILES = HERE / "Lab_Files"
if not LAB_FILES.exists():
    LAB_FILES = HERE.parent / "Lab_Files"

os.chdir(LAB_FILES)
labs_path = str(LAB_FILES / "labs")
if labs_path not in sys.path:
    sys.path.insert(0, labs_path)

print(f"Working directory: {Path.cwd()}")
print(f"Using lab helpers from: {labs_path}")

## 1. Import Lab Helpers

In [ ]:
from pyspark.sql import functions as F

from day7_common import LAKE_DIR, OUTPUT_DIR, cleaned_orders, ensure_output_dirs, quality_checked_orders, read_bronze_orders, require_source_data, spark_session, write_csv_dir

## 2. Start Spark and Prepare Checked Rows

In [ ]:
require_source_data()
ensure_output_dirs()
spark = spark_session("Day7Notebook07QualityQuarantine")

checked = quality_checked_orders(cleaned_orders(read_bronze_orders(spark)))

## 3. Inspect Quality Results

In [ ]:
checked.select("event_id", "order_id", "is_valid", "quality_errors").orderBy("event_id").show(20, truncate=False)
summary = checked.groupBy("is_valid").count().orderBy("is_valid")
summary.show(truncate=False)

## 4. Split Valid and Invalid Rows

In [ ]:
valid = checked.filter(F.col("is_valid"))
invalid = checked.filter(~F.col("is_valid"))

## 5. Write Silver and Quarantine Outputs

In [ ]:
valid_path = LAKE_DIR / "silver" / "orders_valid"
quarantine_path = LAKE_DIR / "quarantine" / "orders_invalid"
valid.write.mode("overwrite").parquet(str(valid_path))
invalid.write.mode("overwrite").parquet(str(quarantine_path))

## 6. Preview Quarantine and Write Summaries

In [ ]:
invalid_preview = invalid.select(
    "event_id",
    "order_id",
    F.concat_ws("|", F.col("quality_errors")).alias("quality_errors"),
).orderBy("event_id")
invalid_preview.show(truncate=False)

write_csv_dir(summary, OUTPUT_DIR / "notebook_07_quality_summary")
write_csv_dir(invalid_preview, OUTPUT_DIR / "notebook_07_quarantine_preview")
print(f"Valid rows: {valid.count()}; quarantined rows: {invalid.count()}")

## Clean Shutdown

Stop the SparkSession when you are done with the notebook. The next notebook will create its own session.

In [ ]:
# Run this at the end of the notebook, or before restarting/rerunning the lab.
spark.stop()